In [ ]:
from official_libraries import *
from utils import *
import os
from scipy.signal import find_peaks
from scipy.signal import argrelextrema
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')

%matplotlib qt

# General

In [ ]:
folder = ""
folder_processed = ""

## Dictionary subjects

In [ ]:
SUBJECTS = {
    1: dict(
        p2=4, p1=5,
        folder=os.path.join(folder, 'subj_158/'),
        weight=67, sex='F',
        ex_order=['side', 'sit2stand', 'crouched', 'lift_2hands', 'lift_1hand'],
        path=os.path.join(folder_processed, 'subj_158/'),
        note='opposite'
    ),

    2: dict(
        p2=5, p1=2,
        folder=os.path.join(folder, 'subj_633/'),
        weight=70, sex='M',
        ex_order=['sit2stand', 'lift_1hand', 'crouched', 'side', 'lift_2hands'],
        path=os.path.join(folder_processed, 'subj_633/'),
        note='same'
    ),

    3: dict(
        p2=2, p1=5,
        folder=os.path.join(folder, 'subj_906/'),
        weight=70, sex='M',
        ex_order=['crouched', 'lift_2hands', 'side', 'sit2stand', 'lift_1hand'],
        path=os.path.join(folder_processed, 'subj_906/'),
        note='same'
    ),

    4: dict(
        p2=4, p1=1,
        folder=os.path.join(folder, 'subj_958/'),
        weight=63, sex='F',
        ex_order=['lift_1hand', 'sit2stand', 'side', 'lift_2hands', 'crouched'],
        path=os.path.join(folder_processed, 'sub_958/'),
        note='same'
    ),

    5: dict(
        p2=4, p1=2,
        folder=os.path.join(folder, 'sub_127/'),
        weight=75, sex='M',
        ex_order=['crouched', 'lift_1hand', 'side', 'lift_2hands', 'sit2stand'],
        path=os.path.join(folder_processed, 'subj_127/'),
        note='opposite'
    ),

    6: dict(
        p2=1, p1=4,
        folder=os.path.join(folder, 'subj_98/'),
        weight=72, sex='M',
        ex_order=['lift_2hands', 'sit2stand', 'crouched', 'lift_1hand', 'side'],
        path=os.path.join(folder_processed, 'subj_98/'),
        note='same'
    ),

    7: dict(
        p2=5, p1=2,
        folder=os.path.join(folder, 'subj_547/'),
        weight=70, sex='F',
        ex_order=['sit2stand', 'lift_1hand', 'crouched', 'side', 'lift_2hands'],
        path=os.path.join(folder_processed, 'subj_547/'),
        note='opposite'
    ),

    8: dict(
        p2=2, p1=2,
        folder=os.path.join(folder, 'subj_815/'),
        weight=68, sex='M',
        ex_order=['lift_1hand', 'lift_2hands', 'side', 'sit2stand', 'crouched'],
        path=os.path.join(folder_processed, 'subj_815/'),
        note='opposite'
    ),

    9: dict(
        p2=1, p1=3,
        folder=os.path.join(folder, 'subj_914/'),
        weight=55, sex='F',
        ex_order=['lift_2hands', 'sit2stand', 'lift_1hand', 'crouched', 'side'],
        path=os.path.join(folder_processed, 'subj_914/'),
        note='opposite'
    ),

    10: dict(
        p2=5, p1=2,
        folder=os.path.join(folder, 'subj_971/'),
        weight=63, sex='F',
        ex_order=['side', 'lift_1hand', 'sit2stand', 'crouched', 'lift_2hands'],
        path=os.path.join(folder_processed, 'subj_971/'),
        note='same'
    ),

    11: dict(
        p2=1, p1=3,
        folder=os.path.join(folder, 'subj_279/'),
        weight=90, sex='M',
        ex_order=['lift_2hands', 'crouched', 'lift_1hand', 'side', 'sit2stand'],
        path=os.path.join(folder_processed, 'subj_279/'),
        note='opposite'
    ),

    12: dict(
        p2=3, p1=1,
        folder=os.path.join(folder, 'subj_965/'),
        weight=58, sex='F',
        ex_order=['lift_1hand', 'side', 'lift_2hands', 'crouched', 'sit2stand'],
        path=os.path.join(folder_processed, 'subj_965/'),
        note='opposite'
    ),
}


## Run a subject: e.g. subj 98

In [ ]:
s = 6
cfg = SUBJECTS[s]

p1 = cfg['p1']
p2 = cfg['p2']
folder = cfg['folder']
weight = cfg['weight']
sex = cfg['sex']
ex_order = cfg['ex_order']
path = cfg['path']

### Unimanual

In [ ]:
ex = 'ex{}'.format(p1)
data = lab_data(folder+'segmented/', ex, weight, sex)
data.load_stereo_file()
data.load_pedana_file(show=False,apply_rotation_rf=True)
data.same_length()
data.com_trajectory(show=False)
data.joint_angles_(show=False)
imu = imu_data(path+"lab/".format(ex_order[p1-1]), ex_order[p1-1])
imu.load_imu_file()

com_vt = interpolation(data.com['y'], mask=300) 
com_vt_diff = np.diff(com_vt) * 200
com_vt_diff_abs = abs(com_vt_diff)

peaks, val = find_peaks(com_vt_diff_abs, height=max(com_vt_diff_abs)/3) 

peaks_imu = [p//2 for p in peaks]
mag = (imu.t['ax']**2+imu.t['ay']**2+imu.t['az']**2).values

local_minima = argrelextrema(np.array(com_vt_diff_abs), np.less)[0]

def find_nearest(array, value):
    idx = np.argmin(np.abs(array - value))
    return idx

descend_peaks = peaks[::2]
ascend_peaks = peaks[1::2]

init_ix, ending_ix = [], []
init, ending = [], []
for d, a in zip(descend_peaks, ascend_peaks):
    if local_minima[find_nearest(local_minima, d)] > d:
        d_ix = find_nearest(local_minima, d) -1
    else:
        d_ix = find_nearest(local_minima, d)
    init_ix.append(d_ix)
    init.append(local_minima[d_ix])
    if local_minima[find_nearest(local_minima, a)] < a:
        a_ix = find_nearest(local_minima, a) +1
    else:
        a_ix = find_nearest(local_minima, a)
    ending_ix.append(a_ix)
    ending.append(local_minima[a_ix])


points = {'1':[], '2':[],'3':[],'4':[],'5':[]}
reps = ['1', '2','3','4','5']
r = reps
for c, i, e in zip(r, init_ix, ending_ix):
    points['{}'.format(c)] = local_minima[i:e+1]

angles = {}
imu_ = {}
for c, i, e in zip(r, init, ending):
    angles['{}'.format(c)]={'trunk':data.angles['trunk'][i:e].values, 'com_vt':com_vt[i:e].values, 'com_ap':data.com['z'][i:e].values, 
                            'cop_ap':data.pedana['zcop'][i:e].values,'cop_ml':data.pedana['xcop'][i:e].values,
                            'fx':data.pedana['fx'][i:e].values,'fy':data.pedana['fy'][i:e].values,'fz':data.pedana['fz'][i:e].values,
                            'rmal':data.stereo[['rmal_x', 'rmal_y', 'rmal_z']][i:e].values, 
                            'lmal':data.stereo[['lmal_x', 'lmal_y', 'lmal_z']][i:e].values,
                            'rtoe':data.stereo[['rtoe_x', 'rtoe_y', 'rtoe_z']][i:e].values, 
                            'ltoe':data.stereo[['ltoe_x', 'ltoe_y', 'ltoe_z']][i:e].values,
                            'rheel':data.stereo[['rheel_x', 'rheel_y', 'rheel_z']][i:e].values, 
                            'lheel':data.stereo[['lheel_x', 'lheel_y', 'lheel_z']][i:e].values}
    if s == 9:
        angles['{}'.format(c)]['knee_left']=data.angles['knee_left'][i:e].values #9
    else:
        angles['{}'.format(c)]['knee_right']=data.angles['knee_right'][i:e].values #9

    angles['{}'.format(c)]['imu'] = {'acc':imu.t[['ax', 'ay','az']].iloc[i//2:e//2],
                                     'gyro':imu.t[['gx', 'gy','gz']].iloc[i//2:e//2]}
    
    middle = local_minima[(local_minima > i) & (local_minima < e)][0]                                            

angles['points'] = points  

angles['whole_trunk'] = {'acc':imu.t[['ax', 'ay','az']],
                        'gyro':imu.t[['gx', 'gy','gz']]}

with open('/Volumes/Seagate/biomech_analysis/clustering/curves/1_angles_s'+ '{}'.format(s) + '.pkl', 'wb') as handle:
    pickle.dump(angles, handle, protocol=pickle.HIGHEST_PROTOCOL) 

### Bimanual

In [ ]:
ex = 'ex{}'.format(p1)
data = lab_data(folder+'segmented/', ex, weight, sex)
data.load_stereo_file()
data.load_pedana_file(show=False,apply_rotation_rf=True)
data.same_length()
data.com_trajectory(show=False)
data.joint_angles_(show=False)
imu = imu_data(path+"lab/".format(ex_order[p1-1]), ex_order[p1-1])
imu.load_imu_file()

com_vt = interpolation(data.com['y'], mask=300) 
com_vt_diff = np.diff(com_vt) * 200
com_vt_diff_abs = abs(com_vt_diff)

peaks, val = find_peaks(com_vt_diff_abs, height=max(com_vt_diff_abs)/3) 

peaks_imu = [p//2 for p in peaks]
mag = (imu.t['ax']**2+imu.t['ay']**2+imu.t['az']**2).values

local_minima = argrelextrema(np.array(com_vt_diff_abs), np.less)[0]

def find_nearest(array, value):
    idx = np.argmin(np.abs(array - value))
    return idx

descend_peaks = peaks[::2]
ascend_peaks = peaks[1::2]

init_ix, ending_ix = [], []
init, ending = [], []
for d, a in zip(descend_peaks, ascend_peaks):
    if local_minima[find_nearest(local_minima, d)] > d:
        d_ix = find_nearest(local_minima, d) -1
    else:
        d_ix = find_nearest(local_minima, d)
    init_ix.append(d_ix)
    init.append(local_minima[d_ix])
    if local_minima[find_nearest(local_minima, a)] < a:
        a_ix = find_nearest(local_minima, a) +1
    else:
        a_ix = find_nearest(local_minima, a)
    ending_ix.append(a_ix)
    ending.append(local_minima[a_ix])


points = {'1':[], '2':[],'3':[],'4':[],'5':[]}
reps = ['1', '2','3','4','5']
r = reps
for c, i, e in zip(r, init_ix, ending_ix):
    points['{}'.format(c)] = local_minima[i:e+1]

angles = {}
imu_ = {}
for c, i, e in zip(r, init, ending):
    angles['{}'.format(c)]={'trunk':data.angles['trunk'][i:e].values, 'com_vt':com_vt[i:e].values, 'com_ap':data.com['z'][i:e].values, 
                            'cop_ap':data.pedana['zcop'][i:e].values,'cop_ml':data.pedana['xcop'][i:e].values,
                            'fx':data.pedana['fx'][i:e].values,'fy':data.pedana['fy'][i:e].values,'fz':data.pedana['fz'][i:e].values,
                            'rmal':data.stereo[['rmal_x', 'rmal_y', 'rmal_z']][i:e].values, 
                            'lmal':data.stereo[['lmal_x', 'lmal_y', 'lmal_z']][i:e].values,
                            'rtoe':data.stereo[['rtoe_x', 'rtoe_y', 'rtoe_z']][i:e].values, 
                            'ltoe':data.stereo[['ltoe_x', 'ltoe_y', 'ltoe_z']][i:e].values,
                            'rheel':data.stereo[['rheel_x', 'rheel_y', 'rheel_z']][i:e].values, 
                            'lheel':data.stereo[['lheel_x', 'lheel_y', 'lheel_z']][i:e].values}
    if s == 9:
        angles['{}'.format(c)]['knee_left']=data.angles['knee_left'][i:e].values #9
    else:
        angles['{}'.format(c)]['knee_right']=data.angles['knee_right'][i:e].values #9

    angles['{}'.format(c)]['imu'] = {'acc':imu.t[['ax', 'ay','az']].iloc[i//2:e//2],
                                     'gyro':imu.t[['gx', 'gy','gz']].iloc[i//2:e//2]}
    
    middle = local_minima[(local_minima > i) & (local_minima < e)][0]                                            

angles['points'] = points  

angles['whole_trunk'] = {'acc':imu.t[['ax', 'ay','az']],
                        'gyro':imu.t[['gx', 'gy','gz']]}

with open('/Volumes/Seagate/biomech_analysis/clustering/curves/1_angles_s'+ '{}'.format(s) + '.pkl', 'wb') as handle:
    pickle.dump(angles, handle, protocol=pickle.HIGHEST_PROTOCOL) 

# Subjects Needing Manual Adjustments

The following (subject, task) pairs required manual intervention during
COM vertical velocity–based segmentation due to spurious peak detection
or local minima misidentification.

**Format**  
`(subject, task) : description of adjustment`

### Peak-related adjustments
- **(1, 1)**: remove peaks `[0, 1, 6, 7, 8, 9]`
- **(1, 3)**: remove peaks `[0, 1, 2, 3, 7, 11, 12, 13]`
- **(1, 10)**: remove peak `[2]`
- **(1, 12)**: remove peaks `[0, 1, 2, 3, 4, 7]`
- **(2, 1)**: remove peaks `[0, 1]`
- **(2, 10)**: remove peaks `[0, 1]`
- **(2, 12)**: remove peaks `[2, 3]`

### Local-minima–related adjustments
- **(1, 5)**: remove local minimum `[14]`
- **(2, 2)**: remove local minimum `[17]`

> **Note**  
> These adjustments were applied only in the listed cases and do not
> affect the general preprocessing pipeline. All other subjects and
> tasks were processed automatically using the same parameters.


In [ ]:

# ---> Adjustments on peaks
# ... 
peaks, val = find_peaks(com_vt_diff_abs, height=max(com_vt_diff_abs)/3) 
# peaks = np.delete(peaks, [0,1, 6, 7,8,9]) # unimanual, subj 1
# peaks = np.delete(peaks, [0,1,2,3, 7,11,12,13]) #unimanual, subj 3
# peaks = np.delete(peaks, 2) #unimanual, subj 10, bimanual, subj 5
# peaks = np.delete(peaks, [0,1,2,3,4,7]) #unimanual, subj 12
# peaks = np.delete(peaks, [0,1]) #bimanual, subj 1, bimanual, subj 10
# peaks = np.delete(peaks, [2,3]) #bimanual, subj 12


# ---> Adjustments on local minima
# ...
local_minima = argrelextrema(np.array(com_vt_diff_abs), np.less)[0]

import numpy as np
def find_nearest(array, value):
    idx = np.argmin(np.abs(array - value))
    return idx

descend_peaks = peaks[::2]
ascend_peaks = peaks[1::2]

# local_minima = np.delete(local_minima, 14) #unimanual, subj 5 
# local_minima = np.delete(local_minima, 17) #bimanual, subj 2